# Traffic assignment

## Setup

### Get data

In [1]:
from aequilibrae import Project

p = Project()
p.open("/mnt/c/Users/TylerPearn/Documents/chicago_sample_mode2_clean/chicago_sample_model")
p.network.links.data

,link_id,a_node,b_node,direction,distance,modes,link_type,name,lanes_ab,lanes_ba,capacity_ab,capacity_ba,speed_ab,speed_ba,geometry,free_flow_time,b,power
0,1792,14791,23462,1,906.851355,c,default,NaN,None,None,720.0,NaN,25.0,None,"LINESTRING (-87.62495 41.60049, -87.63244 41.6...",1.272,0.15,4
1,1793,14791,23463,1,650.245013,c,default,NaN,None,None,1728.0,NaN,32.0,None,"LINESTRING (-87.62495 41.60049, -87.61715 41.6...",0.703,0.15,4
2,1794,14791,25973,1,460.799966,c,default,NaN,None,None,1728.0,NaN,32.0,None,"LINESTRING (-87.62495 41.60049, -87.63047 41.6...",0.429,0.15,4
3,1795,14792,14821,1,945.395541,c,default,NaN,None,None,924.0,NaN,38.0,None,"LINESTRING (-88.28207 41.78472, -88.27071 41.7...",0.900,0.15,4
4,1796,14792,17238,1,276.017514,c,default,NaN,None,None,1944.0,NaN,36.0,None,"LINESTRING (-88.28207 41.78472, -88.28171 41.7...",0.255,0.15,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37737,41320,2591,19719,0,784.290551,c,centroid_connector,centroid connector zone 2591,None,None,119998.8,119998.8,NaN,None,"LINESTRING (-88.01086 42.53881, -88.00134 42.5...",0.001,0.15,4
37738,41321,2594,19455,0,3506.781508,c,centroid_connector,centroid connector zone 2594,None,None,119998.8,119998.8,NaN,None,"LINESTRING (-88.5032 42.525, -88.51548 42.49477)",0.001,0.15,4
37739,41322,2595,19829,0,856.605706,c,centroid_connector,centroid connector zone 2595,None,None,119998.8,119998.8,NaN,None,"LINESTRING (-88.34923 42.55372, -88.34679 42.5...",0.001,0.15,4
37740,41323,2641,22875,0,1322.527509,c,centroid_connector,centroid connector zone 2641,None,None,119998.8,119998.8,NaN,None,"LINESTRING (-87.90777 42.62949, -87.91569 42.6...",0.001,0.15,4


In [2]:
# p.network.links.data["link_id" == 21615]
p.network.links.data.loc[p.network.links.data['link_id'] == 21615]

,link_id,a_node,b_node,direction,distance,modes,link_type,name,lanes_ab,lanes_ba,capacity_ab,capacity_ba,speed_ab,speed_ba,geometry,free_flow_time,b,power
19720,21615,21102,15852,1,2865.741485,c,default,NaN,None,None,9600.0,NaN,55.0,None,"LINESTRING (-87.63019 41.81387, -87.63064 41.8...",1.66,0.15,4


### Inspect available modes

In [3]:
p.network.modes.data

,mode_name,mode_id,description
0,car,c,All motorized vehicles
1,low,l,Motorised vehicles with low toll road cost
2,medium,m,Motorised vehicles with medium toll road cost
3,high,h,Motorised vehicles with high toll road cost


Pick some modes and build the graph.

In [4]:
p.network.build_graphs(modes=['c'])
graph = p.network.graphs['c']
graph.compact_graph

,link_id,a_node,b_node,direction,id
0,39019,0,10504,1,0
1,39020,1,10499,1,1
2,39021,2,11645,1,2
3,39022,3,3821,1,3
4,39023,4,4002,1,4
...,...,...,...,...,...
39705,39015,13201,13202,1,39705
39706,39612,13202,593,-1,39706
39707,39016,13202,12803,1,39707
39708,39017,13202,12804,1,39708


Shortest paths, for example by the free flow travel time, can be found.

In [5]:
graph.set_graph("free_flow_time")
res = graph.compute_path(1,8)
res.path

array([39019, 28040, 28060, 28103, 28106, 28108, 28149, 28083, 28016,
       17359, 17139, 17130, 39007, 17162, 17174, 11866, 11861, 11870,
       11877, 11881,  2610,  2609,  2926, 27474,  2915, 27467,  2907,
        2372,  2364,  2904,  2887,  2877,  2861,  2855,  2852,  2849,
        2706,  2707,  2845, 17020,  2844,  2831,  2837,  2827, 36191,
       36179, 36455,  6208,  8476,  6378, 19789, 36350, 36272, 27740,
       18511,  5706,  3255,  3312, 17691, 17692,  2974, 27194,  2965,
        2958,  3653,  3047,  3049, 25582, 25583, 27505, 27507, 37200,
       33583, 37183, 33569,  7172,  2174, 33426,  7136,  7121,  7109,
        2054, 39026])

## Make the traffic assignment

Choose some traffic classes, get their demand matrix and construct the traffic class with the graph and the demand matrix. Then, the traffic assignment object can be made with the chosen traffic class.

In [6]:
from aequilibrae import TrafficAssignment, TrafficClass
from aequilibrae.paths.vdf import bpr

print(p.matrices.list())
am_mat = p.matrices.get_matrix("am.omx", subset=['car', 'heavy_trucks', 'medium_trucks'])
car_class = TrafficClass("car", graph, am_mat)

ta = TrafficAssignment(p)

# Add traffic classes
ta.set_classes([car_class])

ta.set_vdf(bpr, {"alpha": "b", "beta": "power"})
ta.set_capacity_field("capacity")
ta.set_time_field("free_flow_time")
ta.set_algorithm("bfw")
ta.max_iter = 10
ta.rgap_target = 1e-6

                           procedure procedure_id timestamp description
file_name                                                              
am.omx                          None         None      None        None
example_doc.omx                 None         None      None        None
md.omx                          None         None      None        None
odme_outer_iteration_1.omx      None         None      None        None
op.omx                          None         None      None        None
pm.omx                          None         None      None        None


In [7]:
ta.execute()
ta.results()

Equilibrium Assignment                            :   0%|          | 0/10 [00:00<?, ?it/s]

,car_ab,car_ba,car_tot,heavy_trucks_ab,heavy_trucks_ba,heavy_trucks_tot,medium_trucks_ab,medium_trucks_ba,medium_trucks_tot,Preload_AB,...,Congested_Time_Max,Delay_factor_AB,Delay_factor_BA,Delay_factor_Max,VOC_AB,VOC_BA,VOC_max,PCE_AB,PCE_BA,PCE_tot
link_id,,,,,,,,,,,,,,,,,,,,,
1792,555.555285,0.0,555.555285,0.260792,0.0,0.260792,22.227359,0.0,22.227359,0.0,...,1.351267,1.062317,0.0,1.062317,0.802838,0.000000,0.802838,578.043435,0.0,578.043435
1793,1290.456570,0.0,1290.456570,6.000000,0.0,6.000000,57.651682,0.0,57.651682,0.0,...,0.742764,1.056563,0.0,1.056563,0.783627,0.000000,0.783627,1354.108252,0.0,1354.108252
1794,994.093621,0.0,994.093621,3.620590,0.0,3.620590,51.108278,0.0,51.108278,0.0,...,0.437733,1.020357,0.0,1.020357,0.606957,0.000000,0.606957,1048.822489,0.0,1048.822489
1795,1185.063356,0.0,1185.063356,22.406476,0.0,22.406476,54.796763,0.0,54.796763,0.0,...,1.370165,1.522405,0.0,1.522405,1.366089,0.000000,1.366089,1262.266594,0.0,1262.266594
1796,1042.332822,0.0,1042.332822,17.234482,0.0,17.234482,66.492911,0.0,66.492911,0.0,...,0.259306,1.016887,0.0,1.016887,0.579249,0.000000,0.579249,1126.060215,0.0,1126.060215
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41320,576.000000,140.0,716.000000,48.000000,50.0,98.000000,226.000000,188.0,414.000000,0.0,...,0.001000,1.000000,1.0,1.000000,0.007083,0.003150,0.007083,850.000000,378.0,1228.000000
41321,468.000000,78.0,546.000000,30.000000,16.0,46.000000,54.000000,52.0,106.000000,0.0,...,0.001000,1.000000,1.0,1.000000,0.004600,0.001217,0.004600,552.000000,146.0,698.000000
41322,498.000000,54.0,552.000000,8.000000,12.0,20.000000,66.000000,62.0,128.000000,0.0,...,0.001000,1.000000,1.0,1.000000,0.004767,0.001067,0.004767,572.000000,128.0,700.000000


In [8]:
ta.results().columns


Index(['car_ab', 'car_ba', 'car_tot', 'heavy_trucks_ab', 'heavy_trucks_ba',
       'heavy_trucks_tot', 'medium_trucks_ab', 'medium_trucks_ba',
       'medium_trucks_tot', 'Preload_AB', 'Preload_BA', 'Preload_tot',
       'Congested_Time_AB', 'Congested_Time_BA', 'Congested_Time_Max',
       'Delay_factor_AB', 'Delay_factor_BA', 'Delay_factor_Max', 'VOC_AB',
       'VOC_BA', 'VOC_max', 'PCE_AB', 'PCE_BA', 'PCE_tot'],
      dtype='str')

The following utility can be used to graph the output. 

In [9]:
from pandas import DataFrame
from lonboard import Map, PathLayer, PolygonLayer, ScatterplotLayer
from shapely.geometry import box
import geopandas as gpd

import ipywidgets as widgets
from IPython.display import display

"""Graphs the links with the width of each link varying by initial_data_column. If varying_cols is non-empty,
it allows the interactive selection of summing some of the varying_cols. Note that the widths are scaled by the
maximum of the initial column data, so the optional columns must have the same units as the initial columns,
and be a similar order of magnitude, otherwise they might not be visible.

initial_data_col
assignment_links_varying_by(ta.results(), p, "PCE_tot", ["car_tot", "medium_trucks_tot", "heavy_trucks_tot"])

"""
def assignment_links_varying_by(ta_results: DataFrame, project: Project, initial_data_col:str, varying_cols:list[str], separate_routes: bool = True):
    # if "mask" in routes.columns:
    #     routes = routes[routes["mask"] == 1]
    # if "cost" in routes.columns:
    #     routes = routes.sort_values(by="cost", ascending=False)  # sort descending
    links = project.network.links.data
    nodes = project.network.nodes.data

    # route_links_ids = reduce(np.union1d, routes["route set"].abs())
    # route_links = links[links["link_id"].isin(route_links_ids)]
    # separate_routes_ids = list(routes["route set"].abs())
    # separate_routes_links = [links[links["link_id"].isin(ids)] for ids in separate_routes_ids]

    hover_links = links[
        [
            "link_id",
            "link_type",
            "direction",
            "geometry",
        ]
    ].rename(
        columns={
            "link_id": "Link ID",
            "link_type": "Link type",
            "direction": "Direction",
        }
    )

    # link_layer = PathLayer.from_geopandas(
    #         hover_links,
    #         width_units="pixels",
    #         width_min_pixels=0,
    #         get_width=1,
    #         # width_max_pixels=8,
    #         auto_highlight=True,
    #     )

    # extension = DataFilterExtension()
    max_flow = max(ta_results[initial_data_col])
    # print(ta_results[initial_data_col])
    # print("max_flow", max_flow)
    # print("max_flow", max_flow)
    # print("max_flow", max_flow)
    # print("max_flow", max_flow)
    # print("max_flow", max_flow)
    max_width = 16
    offset = 0
    flow_layer = PathLayer.from_geopandas(
        hover_links,
        get_width=max_width*ta_results[initial_data_col]/max_flow + offset,
        width_units = 'pixels',
        get_color=[0,0,255],
    )

    # Multi-select widget to pick columns
    col_selector = widgets.SelectMultiple(
        options=varying_cols,
        value=varying_cols,
        description="Sum Columns:",
    )


    # Update only the width attribute when selection changes
    def update_width(change):
        selected = list(change["new"])
        if selected:
            raw_sum = ta_results[selected].sum(axis=1)
            flow_layer.get_width = max_width*raw_sum/max_flow + offset
        else:
            flow_layer.get_width = 100  # Fallback width if nothing is selected

    col_selector.observe(update_width, names="value")

    nodes_layer = ScatterplotLayer.from_geopandas(
        nodes[nodes["is_centroid"] != 1][
            [
                "node_id",
                "geometry",
            ]
        ],
        get_fill_color=[255, 0, 0],
        get_radius=10,
    )
    centroid_layer = ScatterplotLayer.from_geopandas(
        nodes[nodes["is_centroid"] == 1][
            [
                "node_id",
                "geometry",
            ]
        ],
        get_fill_color=[0, 255, 0],
        get_radius=20,
    )

    # 139.45510,-20.60647,139.57898,-20.79793
    bbox = PolygonLayer.from_geopandas(
        gpd.GeoDataFrame(geometry=[box(139.45510, -20.79793, 139.57898, -20.60647)], crs="EPSG:4326"),
        get_fill_color=[100, 100, 100, 128],
    )

    traffic_map = Map(
        # layers=[bbox, link_layer, flow_layer, nodes_layer, centroid_layer],
        layers=[bbox, flow_layer, nodes_layer, centroid_layer],
        # layers=[bbox, link_layer, nodes_layer, centroid_layer],
        basemap=None,
        height="1200px",
        show_side_panel=False,
        show_tooltip=True,
    )
    display(col_selector, traffic_map)

    # return traffic_map

assignment_links_varying_by(ta.results(), p, "PCE_tot", ["car_tot", "medium_trucks_tot", "heavy_trucks_tot"])


SelectMultiple(description='Sum Columns:', index=(0, 1, 2), options=('car_tot', 'medium_trucks_tot', 'heavy_tr…

In [10]:
assignment_links_varying_by(ta.results(), p, "VOC_max", ["VOC_max", "VOC_AB", "VOC_BA"])

SelectMultiple(description='Sum Columns:', index=(0, 1, 2), options=('VOC_max', 'VOC_AB', 'VOC_BA'), value=('V…

# Multi-class assignment
The previous example had one main class, "car", and three types called user classes that had different demands. Aequilibrae also can perform static traffic assignment with multiple classes, that might

In [11]:
p.network.modes.columns

('mode_name', 'mode_id', 'description')

In [12]:
from shapely.geometry import Point
import geopandas as gpd
import pandas as pd

# found by looking at a link in the city
center = Point(-87.63019, 41.81387)
circle = center.buffer(0.1) # 0.1 gets 3700 ish nodes, about 1/10 of the total nodes
# gdf = gpd.GeoDataFrame(
#     {'id': [1]}, geometry=[circle], crs="EPSG:4326"
# )

# circle

rows_new_links = p.network.links.data.sindex.query(circle, predicate="contains")
rows_new_links



# p.network.modes.data.loc[1] = ["low", "low", "Motorised vehicles with low toll road cost"]
# p.network.modes.data.loc[2] = ["medium", "medium", "Motorised vehicles with medium toll road cost"]
# p.network.modes.data.loc[3] = ["high", "high", "Motorised vehicles with high toll road cost"]
# p.network.modes.insert(mode_name = "low", mode_id = "l", description = "Motorised vehicles with low toll road cost")
# p.network.modes.insert(mode_name = "medium", mode_id = "m", description = "Motorised vehicles with medium toll road cost")
# p.network.modes.insert(mode_name = "high", mode_id = "h", description = "Motorised vehicles with high toll road cost")
#  = pd.DataFrame([{"Name": "Charlie", "Age": 35}])
# new_row = pd.DataFrame([{"Name": "Charlie", "Age": 35}])
# new_row = pd.DataFrame([{"Name": "Charlie", "Age": 35}])


# p.network.modes.data = pd.concat([df, pd.DataFrame([{"mode_name": "low_cost", "mode_id": "low", "description": "Motorised vehicles with low toll road cost"}])], ignore_index=True)


# # graph = p.network.graphs['c']








array([ 1880,  1886,  1242, ..., 17889, 36318, 14415], shape=(3811,))

In [13]:
p.network.build_graphs(modes=['c'])
graph = p.network.graphs['c']

In [14]:
df = graph.network


low_cost = 0
medium_cost = 1000
high_cost = 1000000
# one column
df["low"] = low_cost * df["link_id"].isin(rows_new_links)
df["medium"] = medium_cost * df["link_id"].isin(rows_new_links)
df["high"] = high_cost * df["link_id"].isin(rows_new_links)
# df["mod3"] = df["link_id"] % 3

graph.prepare_graph(graph.centroids)

am_mat["low"] = am_mat["car"] / 3
am_mat["medium"] = am_mat["car"] / 3
am_mat["high"] = am_mat["car"] / 3

# low_cost_class = TrafficClass("low", graph, am_mat.subset(["low", "medium", "high"]))
# medium_cost_class = TrafficClass("medium", graph, am_mat.subset(["low", "medium", "high"]))
# high_cost_class = TrafficClass("high", graph, am_mat.subset(["low", "medium", "high"]))

low_cost_class = TrafficClass("low", graph, am_mat.subset(["low"]))
medium_cost_class = TrafficClass("medium", graph, am_mat.subset(["medium"]))
high_cost_class = TrafficClass("high", graph, am_mat.subset(["high"]))



# p.network.links = 

In [15]:
ta2 = TrafficAssignment(p)

# Add traffic classes
ta2.set_classes([low_cost_class, medium_cost_class, high_cost_class])

ta2.set_vdf(bpr, {"alpha": "b", "beta": "power"})
ta2.set_capacity_field("capacity")
ta2.set_time_field("free_flow_time")
ta2.set_algorithm("bfw")
ta2.max_iter = 10
ta2.rgap_target = 1e-6

In [16]:
ta2.execute()
results = ta2.results()
results

Equilibrium Assignment                            :   0%|          | 0/10 [00:00<?, ?it/s]

,low_ab,low_ba,low_tot,medium_ab,medium_ba,medium_tot,high_ab,high_ba,high_tot,Preload_AB,...,Congested_Time_Max,Delay_factor_AB,Delay_factor_BA,Delay_factor_Max,VOC_AB,VOC_BA,VOC_max,PCE_AB,PCE_BA,PCE_tot
link_id,,,,,,,,,,,,,,,,,,,,,
1792,183.713187,0.000000,183.713187,183.713187,0.000000,183.713187,183.713187,0.000000,183.713187,0.0,...,1.337508,1.051500,0.0,1.051500,0.765472,0.000000,0.765472,551.139562,0.0,551.139562
1793,427.998018,0.000000,427.998018,427.998018,0.000000,427.998018,427.998018,0.000000,427.998018,0.0,...,0.735146,1.045727,0.0,1.045727,0.743052,0.000000,0.743052,1283.994055,0.0,1283.994055
1794,327.099322,0.000000,327.099322,327.099322,0.000000,327.099322,327.099322,0.000000,327.099322,0.0,...,0.435692,1.015600,0.0,1.015600,0.567881,0.000000,0.567881,981.297965,0.0,981.297965
1795,390.768743,0.000000,390.768743,390.768743,0.000000,390.768743,390.768743,0.000000,390.768743,0.0,...,1.249792,1.388658,0.0,1.388658,1.268730,0.000000,1.268730,1172.306230,0.0,1172.306230
1796,362.811714,0.000000,362.811714,362.811714,0.000000,362.811714,362.811714,0.000000,362.811714,0.0,...,0.258759,1.014741,0.0,1.014741,0.559895,0.000000,0.559895,1088.435142,0.0,1088.435142
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41320,192.000000,46.666667,238.666667,192.000000,46.666667,238.666667,192.000000,46.666667,238.666667,0.0,...,0.001000,1.000000,1.0,1.000000,0.004800,0.001167,0.004800,576.000000,140.0,716.000000
41321,156.000000,26.000000,182.000000,156.000000,26.000000,182.000000,156.000000,26.000000,182.000000,0.0,...,0.001000,1.000000,1.0,1.000000,0.003900,0.000650,0.003900,468.000000,78.0,546.000000
41322,166.000000,18.000000,184.000000,166.000000,18.000000,184.000000,166.000000,18.000000,184.000000,0.0,...,0.001000,1.000000,1.0,1.000000,0.004150,0.000450,0.004150,498.000000,54.0,552.000000


In [17]:
10*results["high_tot"]/50 + 0

link_id
1792     36.742637
1793     85.599604
1794     65.419864
1795     78.153749
1796     72.562343
           ...    
41320    47.733333
41321    36.400000
41322    36.800000
41323    44.533333
41324    43.466667
Name: high_tot, Length: 37742, dtype: float64

# These low medium and high cost don't look different - check plotting code, change costs or move circle.

In [18]:
assignment_links_varying_by(results, p, "low_tot", ["low_tot", "medium_tot", "high_tot"])


SelectMultiple(description='Sum Columns:', index=(0, 1, 2), options=('low_tot', 'medium_tot', 'high_tot'), val…